In [1]:
from pyspark.sql import functions as F
from spark_session import get_spark
import sources

In [2]:
import os
import glob

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [3]:
spark = get_spark("nulls_dupes")
orders = sources.orders(spark)

In [4]:
orders.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in orders.columns]).show()
print("rows: ", orders.count(), "distinct rows: ", orders.distinct().count(),
      "distinct order ids: ", orders.select("order_id").distinct().count())

+--------+--------+-----------+----------+--------+------+----------+
|order_id|order_ts|customer_id|product_id|quantity|status|updated_at|
+--------+--------+-----------+----------+--------+------+----------+
|       0|       0|       5645|         0|       0|     0|         0|
+--------+--------+-----------+----------+--------+------+----------+

rows:  282000 distinct rows:  281300 distinct order ids:  280000


In [5]:
%run cleaning.ipynb

In [6]:
from cleaning import clean_customers, clean_orders

In [7]:
good, rejected = clean_orders(orders, sources.products(spark))
print("good: ", good.count(), "unique:", good.select("order_id").distinct().count())
rejected.groupBy("reject_reason").count().show()
good.groupBy("status").count().orderBy("status").show()

good:  270380 unique: 270380
+-------------+------+
|reject_reason| count|
+-------------+------+
|         NULL|270380|
+-------------+------+

+--------+------+
|  status| count|
+--------+------+
|    paid|202218|
|refunded|  1258|
| shipped| 66904|
+--------+------+



In [8]:
print("customers:", clean_customers(sources.customers(spark)).count())

customers: 2000


In [ ]:
spark.stop()